# Lowest-Performing Run
Finds the lowest-performing completed run, using the metrics from `04_production_metrics`.

In [1]:
import contextlib
import io

import pandas as pd

# Run 04 (which runs 03 and 02) to get the metrics table. Its printed output is hidden here.
with contextlib.redirect_stdout(io.StringIO()):
    get_ipython().run_line_magic("run", "04_production_metrics.ipynb")

## How runs are compared
- The runs make different products with different target rates, so they're compared by target attainment (throughput as a percentage of the target), not raw throughput.
- Only completed runs are compared. `R104` has no end time.

## Ranking, lowest first

In [2]:
completed = metrics[metrics["run_end"].notna()].copy()
# A run's production was counted for the whole run if no note says counting started late or stopped early
partial = completed["data_quality_notes"].str.contains("counted from|counted to")
completed["counted_for_whole_run"] = partial.map({True: "no", False: "yes"})

ranking = completed.sort_values("target_attainment_percent")[[
    "run_id", "equipment_id", "product_code", "cases_produced", "operating_throughput_cph",
    "target_rate_cases_per_hour", "target_attainment_percent", "counted_for_whole_run",
]]
lowest = ranking.iloc[0]
print(f"Lowest-performing completed run: {lowest['run_id']} ({lowest['target_attainment_percent']:.1f}% of target)")
print(f"Notes: {completed.set_index('run_id').loc[lowest['run_id'], 'data_quality_notes']}")

ranking.style.hide(axis="index").format(
    {"operating_throughput_cph": "{:.1f}", "target_attainment_percent": "{:.1f}"}
)

Lowest-performing completed run: R101 (49.5% of target)
Notes: counted to C009 at 14:40, 30 min before the end; readings not used: C010 (glitch: count dropped), C011 (glitch: count dropped); D002 starts before R101: only the overlap counts; D004 shares time with an earlier stop: counted once


run_id,equipment_id,product_code,cases_produced,operating_throughput_cph,target_rate_cases_per_hour,target_attainment_percent,counted_for_whole_run
R101,PKG-01,FLOUR-A,250,297.0,600,49.5,no
R102,PKG-02,FLOUR-B,280,305.5,420,72.7,no
R103,PKG-02,FLOUR-B,290,308.4,420,73.4,yes
R100,PKG-01,FLOUR-A,420,466.7,600,77.8,no


## How sure is this?
- `R101` is lowest by attainment and by raw throughput.
- Its result depends on decision 1 in `03`: the `C010` drop is treated as a glitch, so `R101`'s production after 14:40 isn't counted, but its full operating time is. If `C010` were a counter reset instead, `R101` would reach 91.1% and `R102` (72.7%) would be the lowest.
- `R103` is the only run whose production was counted for the whole run. At 73.4% it's the lowest run that doesn't depend on a reading being left out.